# Notebook 2: Graph Construction

Builds all network representations used in the subsequent analysis notebooks.

**Inputs** from `../data/processed/`:
- `events_{size}_{period}.pkl` and `users_{size}_{period}.pkl` — 16 cleaned DataFrames from NB1

**Outputs** saved to `../data/processed/`:
- `event_graph_{key}.pkl` — event-event LCC graphs with ideology node attributes (for GE analysis)
- `backboned_graph_{key}.pkl` — noise-corrected backbone of event-event LCC (for GE analysis)
- `user_graph_{key}.pkl` — user-user LCC graphs (for demographic mixing analysis)
- `filtered_events_{key}.pkl` — events filtered to LCC membership
- `filtered_users_{key}.pkl` — users connected to LCC events
- `dropped_events_summary.pkl` — dropped event component analysis (Table D.1)
- `dropped_users_summary.pkl` — dropped user component analysis (Table E.1)

**Pipeline**
1. Load cleaned datasets from NB1
2. Build bipartite (event × user) graphs
3. Project to event-event unipartite; extract Largest Connected Component (LCC)
4. Project to user-user unipartite; extract LCC
5. Filter event and user DataFrames to LCC membership
6. Backbone event-event LCC graphs (noise correction; no threshold applied)
7. Save all outputs

> **Runtime note**: Graph construction — especially the user-user projection for medium events — is computationally expensive. Medium-event datasets can take 10–20 minutes per period.

In [1]:
import pickle
import sys
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from networkx.algorithms import bipartite

sys.path.insert(0, str(Path('../src').resolve()))
import backboning as bb

DATA_DIR = Path('../data/processed')

## 1. Load Cleaned Datasets

In [2]:
KEYS = [
    'small_2013-2014', 'small_2014-2015', 'small_2015-2016', 'small_2016-2017',
    'medium_2013-2014', 'medium_2014-2015', 'medium_2015-2016', 'medium_2016-2017',
]

event_datasets = {k: pd.read_pickle(DATA_DIR / f'events_{k}.pkl') for k in KEYS}
user_datasets  = {k: pd.read_pickle(DATA_DIR / f'users_{k}.pkl')  for k in KEYS}

print(f"Loaded {len(KEYS)} event and user datasets")
for k in KEYS:
    print(f"  {k}: {len(event_datasets[k]):,} events, {len(user_datasets[k]):,} users")

Loaded 8 event and user datasets
  small_2013-2014: 8,556 events, 64,651 users
  small_2014-2015: 9,904 events, 73,877 users
  small_2015-2016: 14,172 events, 101,016 users
  small_2016-2017: 17,845 events, 115,214 users
  medium_2013-2014: 4,713 events, 126,884 users
  medium_2014-2015: 5,779 events, 150,725 users
  medium_2015-2016: 7,766 events, 189,671 users
  medium_2016-2017: 7,575 events, 192,145 users


## 2. Build Bipartite Graphs

Each graph has event nodes (`bipartite=0`) and user nodes (`bipartite=1`). Edges connect each event to all of its attendees. Event nodes carry the `weighted_ideology_min_max` attribute used later in the GE analysis.

In [3]:

bipartite_graphs = {}

for key in KEYS:
    events = event_datasets[key]
    users  = user_datasets[key]

    valid_user_ids = set(users['hashed_id'])

    B = nx.Graph()

    # --- Event nodes (bipartite=0) with ideology attributes ---
    for _, ev in events.iterrows():
        B.add_node(
            ev['id'],
            bipartite=0,
            weighted_ideology_min_max=ev['weighted_ideology_min_max'],
            n_attending=ev['n_attending'],
            percent_political=ev['percent_political'],
            owner_id=ev['owner_id'],
        )

    # --- User nodes (bipartite=1) with demographic and ideology attributes ---
    # Add all user nodes upfront so attributes are available in the projection.
    attr_cols = ['hashed_id', 'gender_label', 'heritage_label',
                 'normalized_min_max', 'l_r_min_max', 'political']
    for row in users[attr_cols].to_dict('records'):
        B.add_node(
            row['hashed_id'],
            bipartite=1,
            gender_label=row['gender_label'],
            heritage_label=row['heritage_label'],
            normalized_min_max=row['normalized_min_max'],
            l_r_min_max=row['l_r_min_max'],
            political=bool(row['political']),
        )

    # --- Event-user edges ---
    for _, ev in events.iterrows():
        event_id = ev['id']
        for user_id in ev['attending']:
            if user_id in valid_user_ids:
                B.add_edge(event_id, user_id)

    bipartite_graphs[key] = B
    n_events = sum(1 for _, d in B.nodes(data=True) if d['bipartite'] == 0)
    n_users  = B.number_of_nodes() - n_events
    print(f"{key}: {n_events:,} event nodes, {n_users:,} user nodes, {B.number_of_edges():,} edges")


small_2013-2014: 8,556 event nodes, 64,651 user nodes, 113,402 edges


small_2014-2015: 9,904 event nodes, 73,877 user nodes, 134,534 edges


small_2015-2016: 14,172 event nodes, 101,016 user nodes, 190,173 edges


small_2016-2017: 17,845 event nodes, 115,214 user nodes, 223,218 edges


medium_2013-2014: 4,713 event nodes, 126,884 user nodes, 263,799 edges


medium_2014-2015: 5,779 event nodes, 150,725 user nodes, 328,860 edges


medium_2015-2016: 7,766 event nodes, 189,671 user nodes, 433,814 edges


medium_2016-2017: 7,575 event nodes, 192,145 user nodes, 415,089 edges


## 3. Event-Event Projection and LCC

Project the bipartite graph onto event nodes. Edge weight = number of shared attendees. Retain only the Largest Connected Component.

In [4]:
event_graphs   = {}   # event-event LCC
kept_events    = {}   # event IDs in LCC
kept_users_ee  = {}   # user IDs connected to LCC events (for filtered DataFrames)
dropped_events = {}   # event IDs outside LCC
dropped_users_ee = {} # user IDs connected only to dropped events

for key, B in bipartite_graphs.items():
    print(f"\n{key}")

    event_nodes = [n for n, d in B.nodes(data=True) if d['bipartite'] == 0]

    G_events = bipartite.weighted_projected_graph(B, event_nodes)

    lcc_nodes = max(nx.connected_components(G_events), key=len)
    G_lcc     = G_events.subgraph(lcc_nodes).copy()
    event_graphs[key] = G_lcc

    dropped_event_nodes = set(G_events.nodes()) - lcc_nodes

    # Users connected to kept (LCC) events
    kept_user_nodes = set()
    for ev in lcc_nodes:
        kept_user_nodes.update(B.neighbors(ev))

    # Users connected only to dropped events (not reachable via LCC events)
    dropped_user_nodes = set()
    for ev in dropped_event_nodes:
        dropped_user_nodes.update(B.neighbors(ev))
    dropped_user_nodes -= kept_user_nodes

    kept_events[key]      = lcc_nodes
    kept_users_ee[key]    = kept_user_nodes
    dropped_events[key]   = dropped_event_nodes
    dropped_users_ee[key] = dropped_user_nodes

    print(f"  LCC: {G_lcc.number_of_nodes():,} events, {G_lcc.number_of_edges():,} edges")
    print(f"  Dropped: {len(dropped_event_nodes):,} events, {len(dropped_user_nodes):,} users")


small_2013-2014


  LCC: 8,210 events, 120,395 edges
  Dropped: 346 events, 2,117 users

small_2014-2015


  LCC: 9,603 events, 161,359 edges
  Dropped: 301 events, 1,811 users

small_2015-2016


  LCC: 13,838 events, 255,862 edges
  Dropped: 334 events, 1,807 users

small_2016-2017


  LCC: 17,459 events, 346,228 edges
  Dropped: 386 events, 1,782 users

medium_2013-2014


  LCC: 4,709 events, 372,565 edges
  Dropped: 4 events, 158 users

medium_2014-2015


  LCC: 5,779 events, 542,037 edges
  Dropped: 0 events, 0 users

medium_2015-2016


  LCC: 7,764 events, 812,899 edges
  Dropped: 2 events, 85 users

medium_2016-2017


  LCC: 7,573 events, 691,767 edges
  Dropped: 2 events, 63 users


## 4. User-User Projection and LCC

Project the bipartite graph onto user nodes. Edge weight = number of events co-attended. Retain only the Largest Connected Component.

> This step is the most computationally expensive, particularly for medium-event datasets.

In [5]:
user_graphs   = {}   # user-user LCC
dropped_users_uu = {} # users outside user-user LCC

for key, B in bipartite_graphs.items():
    print(f"\n{key} — projecting user-user graph...")

    user_nodes = [n for n, d in B.nodes(data=True) if d['bipartite'] == 1]

    G_users = bipartite.weighted_projected_graph(B, user_nodes)

    lcc_nodes = max(nx.connected_components(G_users), key=len)
    G_lcc     = G_users.subgraph(lcc_nodes).copy()
    user_graphs[key] = G_lcc

    dropped_users_uu[key] = set(G_users.nodes()) - lcc_nodes

    print(f"  LCC: {G_lcc.number_of_nodes():,} users, {G_lcc.number_of_edges():,} edges")
    print(f"  Dropped: {len(dropped_users_uu[key]):,} users")


small_2013-2014 — projecting user-user graph...


  LCC: 62,534 users, 867,261 edges
  Dropped: 2,117 users

small_2014-2015 — projecting user-user graph...


  LCC: 72,066 users, 1,038,418 edges
  Dropped: 1,811 users

small_2015-2016 — projecting user-user graph...


  LCC: 99,209 users, 1,484,858 edges
  Dropped: 1,807 users

small_2016-2017 — projecting user-user graph...


  LCC: 113,432 users, 1,676,789 edges
  Dropped: 1,782 users

medium_2013-2014 — projecting user-user graph...


  LCC: 126,726 users, 7,463,539 edges
  Dropped: 158 users

medium_2014-2015 — projecting user-user graph...


  LCC: 150,725 users, 9,522,664 edges
  Dropped: 0 users

medium_2015-2016 — projecting user-user graph...


  LCC: 189,586 users, 12,445,290 edges
  Dropped: 85 users

medium_2016-2017 — projecting user-user graph...


  LCC: 192,082 users, 11,783,419 edges
  Dropped: 63 users


## 5. Filter Datasets to LCC Membership

Produce filtered event and user DataFrames aligned with the event-event LCC. These are the datasets used for all downstream statistics and analysis.

In [6]:
filtered_events = {}
filtered_users  = {}

for key in KEYS:
    # Events: keep only those in the event-event LCC
    ev_df = event_datasets[key]
    filtered_events[key] = ev_df[ev_df['id'].isin(kept_events[key])].copy()

    # Users: keep those connected to at least one LCC event
    usr_df = user_datasets[key]
    filtered_users[key] = usr_df[usr_df['hashed_id'].isin(kept_users_ee[key])].copy()

    print(f"{key}: {len(filtered_events[key]):,} events, {len(filtered_users[key]):,} users")

small_2013-2014: 8,210 events, 62,534 users
small_2014-2015: 9,603 events, 72,066 users
small_2015-2016: 13,838 events, 99,209 users
small_2016-2017: 17,459 events, 113,432 users
medium_2013-2014: 4,709 events, 126,726 users
medium_2014-2015: 5,779 events, 150,725 users
medium_2015-2016: 7,764 events, 189,586 users


medium_2016-2017: 7,573 events, 192,082 users


In [7]:

# Build dropped-event and dropped-user summary DataFrames (used in component analysis)

def ideology_shares(event_ids, event_datasets, key):
    ev_df  = event_datasets[key]
    subset = ev_df[ev_df['id'].isin(event_ids)]
    n      = len(subset)
    if n == 0:
        return 0, 0.0, 0.0, 0.0, 0.0
    pct_total = n / len(ev_df)
    pct_l = (subset['weighted_ideology_min_max'] < 0).sum() / n
    pct_r = (subset['weighted_ideology_min_max'] > 0).sum() / n
    pct_n = (subset['weighted_ideology_min_max'] == 0).sum() / n
    return n, pct_total, pct_l, pct_r, pct_n

dropped_events_rows = []
for key, ev_ids in dropped_events.items():
    size, period = key.split('_', 1)
    n, pct_total, pct_l, pct_r, pct_n = ideology_shares(ev_ids, event_datasets, key)
    dropped_events_rows.append({
        'Dataset': key, 'Event Size': size.title(), 'Time Period': period,
        'Dropped Events': n, '% of Total': round(pct_total, 4),
        '% L Events': round(pct_l, 4), '% R Events': round(pct_r, 4),
        '% Neutral Events': round(pct_n, 4),
    })
dropped_events_summary = pd.DataFrame(dropped_events_rows)

dropped_users_rows = []
for key, usr_ids in dropped_users_ee.items():
    size, period = key.split('_', 1)
    usr_df = user_datasets[key]
    subset = usr_df[usr_df['hashed_id'].isin(usr_ids)]
    n      = len(subset)
    n_total = len(usr_df)
    pct_l = (subset['l_r_min_max'] == 'L').sum() / n if n > 0 else 0
    pct_r = (subset['l_r_min_max'] == 'R').sum() / n if n > 0 else 0
    dropped_users_rows.append({
        'Dataset': key, 'Event Size': size.title(), 'Time Period': period,
        'Dropped Users': n, '% of Total': round(n / n_total, 4) if n_total > 0 else 0,
        '% L': round(pct_l, 4), '% R': round(pct_r, 4),
    })
dropped_users_summary = pd.DataFrame(dropped_users_rows)

print("Dropped events summary (Table D.1):")
print(dropped_events_summary.to_string(index=False))
print("\nDropped users summary (Table E.1):")
print(dropped_users_summary.to_string(index=False))


Dropped events summary (Table D.1):
         Dataset Event Size Time Period  Dropped Events  % of Total  % L Events  % R Events  % Neutral Events
 small_2013-2014      Small   2013-2014             346      0.0404      0.5867      0.3960            0.0173
 small_2014-2015      Small   2014-2015             301      0.0304      0.5714      0.4086            0.0199
 small_2015-2016      Small   2015-2016             334      0.0236      0.6138      0.3743            0.0120
 small_2016-2017      Small   2016-2017             386      0.0216      0.6114      0.3756            0.0130
medium_2013-2014     Medium   2013-2014               4      0.0008      0.0000      1.0000            0.0000
medium_2014-2015     Medium   2014-2015               0      0.0000      0.0000      0.0000            0.0000
medium_2015-2016     Medium   2015-2016               2      0.0003      0.0000      0.5000            0.5000
medium_2016-2017     Medium   2016-2017               2      0.0003      0.5000     

## 6. Backbone Event-Event Graphs

Applies noise-corrected backboning (Coscia, 2020) to the event-event LCC graphs. No significance threshold is applied — all edges are retained with their original weights. The backboning step computes noise-corrected scores that inform the topology used in the Generalized Euclidean measure.

In [8]:

def build_backboned_graph(G_lcc):
    """Apply noise-corrected backboning to an event-event LCC graph.

    No threshold is applied — all edges are retained with original weights.
    Node attributes from G_lcc are preserved in the returned graph.
    """
    # Build canonical edge table (src < trg for each undirected edge)
    canonical_rows = [
        {'src': min(u, v), 'trg': max(u, v), 'nij': data.get('weight', 1)}
        for u, v, data in G_lcc.edges(data=True)
    ]
    canonical = pd.DataFrame(canonical_rows)

    # noise_corrected needs both directions to compute per-node strength totals
    reverse    = canonical[['trg', 'src', 'nij']].rename(columns={'trg': 'src', 'src': 'trg'})
    table_both = pd.concat([canonical, reverse], ignore_index=True)

    # Run noise correction; returns canonical edges (src <= trg) with original nij
    bb_scored = bb.noise_corrected(table_both, undirected=True)

    # Build graph directly from scored table — nij is already the original weight
    G_bb = nx.Graph()
    G_bb.add_nodes_from(G_lcc.nodes(data=True))
    for _, row in bb_scored.iterrows():
        G_bb.add_edge(row['src'], row['trg'], weight=row['nij'])

    return G_bb


backboned_graphs = {}

for key, G_lcc in event_graphs.items():
    print(f"{key}...")
    G_bb = build_backboned_graph(G_lcc)
    backboned_graphs[key] = G_bb
    print(f"  {G_bb.number_of_nodes():,} nodes, {G_bb.number_of_edges():,} edges")

print("\nBackboning complete.")


small_2013-2014...


Calculating NC score...


  8,210 nodes, 120,395 edges
small_2014-2015...


Calculating NC score...


  9,603 nodes, 161,359 edges
small_2015-2016...


...

  5,779 nodes, 542,037 edges
medium_2015-2016...


Calculating NC score...


  7,764 nodes, 812,899 edges
medium_2016-2017...


Calculating NC score...


  7,573 nodes, 691,767 edges

Backboning complete.


## 7. Save Outputs

In [9]:
for key in KEYS:
    with open(DATA_DIR / f'event_graph_{key}.pkl',    'wb') as f: pickle.dump(event_graphs[key],    f)
    with open(DATA_DIR / f'backboned_graph_{key}.pkl', 'wb') as f: pickle.dump(backboned_graphs[key], f)
    with open(DATA_DIR / f'user_graph_{key}.pkl',     'wb') as f: pickle.dump(user_graphs[key],     f)
    filtered_events[key].to_pickle(DATA_DIR / f'filtered_events_{key}.pkl')
    filtered_users[key].to_pickle(DATA_DIR / f'filtered_users_{key}.pkl')

dropped_events_summary.to_pickle(DATA_DIR / 'dropped_events_summary.pkl')
dropped_users_summary.to_pickle(DATA_DIR / 'dropped_users_summary.pkl')

print(f"Saved all outputs to {DATA_DIR}")
print("\nFinal graph summary:")
print(f"{'Dataset':<25} {'EE nodes':>9} {'EE edges':>9} {'UU nodes':>9} {'UU edges':>10}")
print("-" * 65)
for key in sorted(KEYS):
    ee = event_graphs[key]
    uu = user_graphs[key]
    print(f"{key:<25} {ee.number_of_nodes():>9,} {ee.number_of_edges():>9,} "
          f"{uu.number_of_nodes():>9,} {uu.number_of_edges():>10,}")

Saved all outputs to ../data/processed

Final graph summary:
Dataset                    EE nodes  EE edges  UU nodes   UU edges
-----------------------------------------------------------------
medium_2013-2014              4,709   372,565   126,726  7,463,539
medium_2014-2015              5,779   542,037   150,725  9,522,664
medium_2015-2016              7,764   812,899   189,586 12,445,290


medium_2016-2017              7,573   691,767   192,082 11,783,419
small_2013-2014               8,210   120,395    62,534    867,261
small_2014-2015               9,603   161,359    72,066  1,038,418
small_2015-2016              13,838   255,862    99,209  1,484,858
small_2016-2017              17,459   346,228   113,432  1,676,789
